In [8]:
from order_book import Order, Trade, OrderBook
from agents import Agent

from collections import deque
from dataclasses import dataclass
from enum import Enum

**Ideas about the order-book**

1. Orderbook is a data structure:
    - has asks and bids, these are dictionary or deques
    - has trade list - a list of trade objects
    - has a timestamp

    - methods
        - we can add to order - this simply adds a limit order to the correct book
        - Matching Engine 
            - this will produce a list of trades with a timestamp
            - if its a market order, it will run through whats in the books, if it runs out hten the order does not go into the books,
            - if its a limit order, then it will partially fill until it can't then be placed in the books in the appropriate place.
            - canel order, removes order from book, currently set so that it pops from start of queue and appends to end for a full cycle (only popping the relevant order, not appending), then returns what it finds.
            - every time a trade is produced or an order is cancelled the timestap of the book increases by one.

    

**Adding the first Agent Tests**

1. Simple test:
    - agent has money on creation 
        - say agent 1 has 1000
        - agent 2 has 0
    - agent 2 then creates an ask for 10 shares at 100
    - agent 1 creates a bid for 10 shares at 100
    - it gets matched, 
        - agent 1 has 0 cash
        - agent 2 has 1000 cash
        - we should include the position of the agent in this too.

***Plan***
1. make the simulation layer.
2. We want something that creates the simulation, it could create the agents, it needs to create the order book,    
    - when we call match order, it returns trades, we should then have this going intoa function: apply_trades
        - apply trades takes in a list of trade objects
        - it then finds the buy_agent_id -> needs to be able to search for it in the simulation object.
        - then it increases the position and decreases the funds appropriates
        - same for the sell id. 
        - loops through all trades in the list

In [ ]:
class ReqType(Enum):
    CANCEL = 'cancel'
    PLACE = 'place'

@dataclass
class Request:
    req_type: ReqType
    order: Order

class Simulation:
    def __init__(self):
        self.book: OrderBook = OrderBook()
        self.agents: dict[int, Agent] = {} # key = agent_id 
        self.request: deque[Request] = deque() # queue of orders waiting to be applied



    def get_requests(self):
        # should loop through the agents and check if any of them want to make a request
        # ignoring the potential latency of each agent, it should get in a list/queue of orders to be executed, then randomise it
        # has the ability to see if the agent wants to cancel their current (limit) order
        pass 

    def apply_request(self):
        # might be apply just one order at a time, then the timestep increases
        # we only do one order at a time so that agents can submit or cancel orders at each time step 
        # they can see what the market looks like
        if not self.requests: 
            return

        req: Request = self.requests.popleft()
        if req.req_type == ReqType.CANCEL:
            self.book.cancel_order(req.order)
        elif req.req_type == ReqType.PLACE:
            trades: list[Trade] = self.book.match_order(req.order)
            self.apply_trades(trades)
        else:
            raise ValueError("Request of invalid type")
        
    def apply_trades(self, trades: list[Trade]):
        # goes through the list of trades, possibly empty
        # can just go until empty applying in any order since its just record keeping.
        for trade in trades:
            # apply to both buyer and seller.
            if trade.buy_agent_id is None:
                raise ValueError("No buyer agent id")
            if trade.sell_agent_id is None:
                raise ValueError("No seller agent id")

            self.agents[trade.buy_agent_id].current_cash -= trade.quantity * trade.price
            self.agents[trade.buy_agent_id].position += trade.quantity
            self.agents[trade.sell_agent_id].current_cash += trade.quantity * trade.price
            self.agents[trade.sell_agent_id].position -= trade.quantity



    def run_sim(self, steps: int):
        # at each step we call get_requests and apply_request
        pass
        

**Upon Creation**